# SE374.R11 - Scientific Research & Publication Benchmark Runner
## 5 Chuyen De Nghiem Thu Khoa Hoc Chuyen Sau (Publication-Grade Benchmarks)

* **Mon hoc:** SE374.R11 - Phat trien phan mem xu ly anh y khoa (UIT - DHQG-HCM)
* **Giang vien huong dan:** TS. Bui Cao Doanh
* **Nhom thuc hien:** [Nhom de xuat rieng - Ho va ten Sinh vien (MSSV)]
* **Kien truc:** **GitHub-First Standalone Scientific Runner** - Tu dong clone ma nguon truc tiep tu GitHub repository (`https://github.com/khoinguyen59/medical-wsss.git`), khong can ket noi Google Drive. Toan bo ket qua nghien cuu duoc tu dong nen va tai truc tiep ve may tinh nguoi dung.
* **5 Thi Nghiem Khoa Hoc Chuyen Sau:**
  1. **Validation-to-Test Scientific Protocol**: Quet sieu tham so tren Val, dong bang va danh gia tren Test (Zero Leakage, mIoU, Dice, FwIoU, Pixel Accuracy, Boundary F1).
  2. **Multi-Magnification & Coarse-to-Fine Adaptive Router**: Mo phong phan giai da tang (5x, 10x, 20x, 40x), router do phuc tap thong minh, duong cong Pareto tiet kiem 80%+ GPU.
  3. **Patient-Level Clinical Biomarker Correlation**: Tinh toan VTP, TSR theo cap do benh nhan, Pearson r, Spearman rho, Bland-Altman va do phu hop nguy co CAP.
  4. **Cross-Dataset LUAD-HistoSeg Generalization**: Thu nghiem tong quat hoa khac co quan (Cross-Organ Zero-Shot tren Ung thu phoi LUAD).
  5. **Whole-ROI 100% Tissue Coverage Evaluation**: Danh gia phan vung toan bo vung mo tren tap Whole-ROI BCSS doi sanh Ground-Truth chuyen gia.

---
## Buoc 1: Dong Bo Ma Nguon Truc Tiep Tu GitHub & Khoi Tao Moi Truong GPU (Zero Google Drive)
O code duoi day tu dong:
1. Clone ma nguon truc tiep tu GitHub: `https://github.com/khoinguyen59/medical-wsss.git` (hoac pull ban moi nhat neu da ton tai).
2. Chuyen vao thu muc du an va nap duong dan he thong vao `sys.path`.
3. Bao toan ABI C-extension cua NumPy/PyTorch/SciPy tren Colab (tu dong khoi phuc neu phien chay truoc bi lech NumPy ABI) va chi cai bo sung cac goi tai du lieu nhe (`gdown`, `huggingface_hub`, `tifffile`).
4. Khoi tao hat giong ngau nhien co dinh (seed 42) va xac thuc GPU (Hoan toan khong can mount Google Drive).

In [ ]:
import os, sys, subprocess, importlib.util
from pathlib import Path

IN_COLAB = "google.colab" in sys.modules
REPO_URL = "https://github.com/khoinguyen59/medical-wsss.git"
REPO_DIR = Path("/content/medical-wsss") if IN_COLAB else Path.cwd().resolve()

if IN_COLAB:
    if not (REPO_DIR / "src" / "pipeline").is_dir():
        print(f"--> Dang clone ma nguon truc tiep tu GitHub: {REPO_URL}...")
        subprocess.run(["git", "clone", "--depth", "1", REPO_URL, str(REPO_DIR)], check=True)
    else:
        print("--> Cap nhat ma nguon moi nhat tu GitHub...")
        subprocess.run(["git", "-C", str(REPO_DIR), "fetch", "origin", "master"], check=False)
        subprocess.run(["git", "-C", str(REPO_DIR), "reset", "--hard", "origin/master"], check=False)

    os.chdir(REPO_DIR)

    # Self-heal if a previous cell execution downgraded NumPy on a NumPy 2.x Colab kernel
    try:
        import numpy as np
        import numpy.random
    except (ValueError, ImportError, RuntimeError, AttributeError, Exception):
        print("--> Phat hien xung dot ABI NumPy tu phien truoc, dang khoi phuc NumPy 2.x tuong thich Colab...")
        subprocess.run([sys.executable, "-m", "pip", "install", "-q", "--prefer-binary", "--no-cache-dir", "--upgrade", "numpy>=2.0.0"], check=False)
        for _k in [k for k in list(sys.modules.keys()) if k == "numpy" or k.startswith("numpy.")]:
            sys.modules.pop(_k, None)
        import numpy as np
        import numpy.random

    missing_pkgs = [
        pkg for mod, pkg in [
            ("gdown", "gdown"),
            ("huggingface_hub", "huggingface_hub"),
            ("tifffile", "tifffile"),
        ]
        if importlib.util.find_spec(mod) is None
    ]
    if missing_pkgs:
        print(f"--> Dang cai dat goi bo tro con thieu: {missing_pkgs}...")
        subprocess.run([sys.executable, "-m", "pip", "install", "-q", "--prefer-binary", "--no-cache-dir", "--no-deps"] + missing_pkgs, check=False)
        subprocess.run([sys.executable, "-m", "pip", "install", "-q", "--prefer-binary", "--no-cache-dir"] + missing_pkgs, check=False)
    for _mod in [k for k in list(sys.modules.keys()) if k == "src" or k.startswith("src.")]:
        sys.modules.pop(_mod, None)

CODE_ROOT = str(REPO_DIR)
if CODE_ROOT not in sys.path:
    sys.path.insert(0, CODE_ROOT)

from src.pipeline.colab_runner import set_reproducible_seed
ENV_INFO = set_reproducible_seed(seed=42, verbose=True)
print(f"[READY] CODE_ROOT = {CODE_ROOT}")
print(f"[READY] Device = {ENV_INFO['device']} | CUDA Available = {ENV_INFO['cuda_available']}")

---
## Buoc 2: Thiet Lap Cau Hinh & Che Do Kiem Dinh (Sanity Dry-Run Toggle)

> **Chu y ve tai nguyen GPU:**
- `SANITY_DRY_RUN = True`: Chay thu nghiem mau nhanh (50 Val, 50 Test, 20 LUAD, 1 Whole-ROI, ~2 phut) de kiem tra tinh dung dan toan bo pipeline truoc khi chay day du.
- `SANITY_DRY_RUN = False`: Chay thuc nghiem toan dien tren 100% du lieu thuc te.

In [ ]:
import torch

SANITY_DRY_RUN = True
DRY_RUN_SAMPLES = 50
USE_AMP = True
AMP_DTYPE = torch.float16

WORK_ROOT = Path("/content" if IN_COLAB else "./se374_scientific_work")
OUTPUT_DIR = WORK_ROOT / "scientific_results"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

print(f"[CONFIG] SANITY_DRY_RUN = {SANITY_DRY_RUN} (Samples per split: {DRY_RUN_SAMPLES})")
print(f"[CONFIG] Device: {ENV_INFO['device']} | AMP: {USE_AMP} ({AMP_DTYPE})")
print(f"[CONFIG] Output Directory: {OUTPUT_DIR}")

---
## Buoc 3: Tai Du Lieu BCSS & Nap Cac Mo Hinh Pretrained (`Stage-1`, `Stage-2`, `Engine`)
Tu dong tai dataset BCSS va ca hai checkpoint chinh thuc (`stage1_checkpoint.pth` va `stage2_checkpoint.pth`).

In [ ]:
from src.colab_pkg.downloader import ensure_bcss_dataset, ensure_checkpoints, resolve_split_dirs
from src.colab_pkg.model_loader import ColabModelLoader, MultiMethodWSSSEngineV2

DATA_ROOT, TEST_IMG_DIR, TEST_MASK_DIR = ensure_bcss_dataset(
    data_root=WORK_ROOT / "BCSS-WSSS",
    zip_path=WORK_ROOT / "BCSS-WSSS.zip",
    extract_dir=WORK_ROOT,
    drive_cache_dir=None,
)
CKPT_PATHS = ensure_checkpoints(
    checkpoint_dir=WORK_ROOT / "checkpoints",
    drive_cache_dir=None,
    require_stage2=True,
)

loader = ColabModelLoader(device=ENV_INFO["device"])
MODEL_STAGE1 = loader.load_stage1_netcam(CKPT_PATHS["stage1"])
MODEL_STAGE2 = loader.load_stage2_deeplabv3plus(CKPT_PATHS["stage2"])
WSSS_ENGINE = MultiMethodWSSSEngineV2(
    model_stage1=MODEL_STAGE1, use_amp=USE_AMP, amp_dtype=AMP_DTYPE,
)
print("[OK] Models loaded successfully: Stage-1 Net_CAM and Stage-2 DeepLabV3+!")

---
## Buoc 4: Thi Nghiem 1 - Quy Trinh Khoa Hoc Validation-to-Test (Zero Patient Leakage)
Quy trinh khoa hoc chuan:
1. Phan tach benh nhan giua Validation va Test triet de theo Patient ID (INVARIANT-08).
2. Quet va chon sieu tham so hau xu ly tren tap Validation (`val_files`).
3. Dong bang sieu tham so da chon va danh gia khach quan tren tap Test (`test_files`).
4. Tinh toan day du: `4-class mIoU`, `mDice`, `FwIoU`, `Pixel Accuracy`, va `Boundary F1`.

In [ ]:
from collections import defaultdict
from src.colab_pkg.downloader import extract_patient_id, resolve_split_dirs
from src.colab_pkg.evaluator import run_val_to_test_scientific_protocol

_, test_img_dir, test_mask_dir = resolve_split_dirs(DATA_ROOT, split="test")
if test_img_dir is None or not test_img_dir.is_dir() or len(list(test_img_dir.glob("*.png"))) == 0:
    test_img_dir, test_mask_dir = TEST_IMG_DIR, TEST_MASK_DIR

_, val_img_dir, val_mask_dir = resolve_split_dirs(DATA_ROOT, split="val")

if val_img_dir is not None and val_img_dir.is_dir() and len(list(val_img_dir.glob("*.png"))) > 0 and val_img_dir != test_img_dir:
    val_files = sorted(val_img_dir.glob("*.png"))
    test_files = sorted(test_img_dir.glob("*.png"))
    val_patient_map = defaultdict(list)
    for p in val_files:
        val_patient_map[extract_patient_id(p.name)].append(p)
    val_pids = sorted(list(val_patient_map.keys()))
    test_patient_map = defaultdict(list)
    for p in test_files:
        test_patient_map[extract_patient_id(p.name)].append(p)
    test_pids = sorted(list(test_patient_map.keys()))
else:
    all_test_patches = sorted(test_img_dir.glob("*.png"))
    patient_map = defaultdict(list)
    for p in all_test_patches:
        patient_map[extract_patient_id(p.name)].append(p)
    unique_pids = sorted(list(patient_map.keys()))
    if len(unique_pids) >= 2:
        n_val_pids = max(1, len(unique_pids) // 3)
        val_pids = sorted(list(unique_pids[:n_val_pids]))
        test_pids = sorted(list(unique_pids[n_val_pids:]))
        val_patient_map = {pid: patient_map[pid] for pid in val_pids}
        test_patient_map = {pid: patient_map[pid] for pid in test_pids}
        val_files = [p for pid in val_pids for p in patient_map[pid]]
        test_files = [p for pid in test_pids for p in patient_map[pid]]
        print(f"[ANTI-LEAKAGE] Phan tach theo Patient ID: {len(val_pids)} benh nhan Val ({len(val_files)} anh) | {len(test_pids)} benh nhan Test ({len(test_files)} anh)")
    else:
        mid = max(1, len(all_test_patches) // 2)
        val_files = all_test_patches[:mid]
        test_files = all_test_patches[mid:] if len(all_test_patches) > 1 else all_test_patches
        val_pids = sorted(list(set(extract_patient_id(p.name) for p in val_files)))
        test_pids = sorted(list(set(extract_patient_id(p.name) for p in test_files)))
        val_patient_map = {pid: [p for p in val_files if extract_patient_id(p.name) == pid] for pid in val_pids}
        test_patient_map = {pid: [p for p in test_files if extract_patient_id(p.name) == pid] for pid in test_pids}
        print(f"[FALLBACK SPLIT] {len(val_files)} anh Val | {len(test_files)} anh Test")
    val_mask_dir = test_mask_dir

if SANITY_DRY_RUN:
    quota_val = max(1, DRY_RUN_SAMPLES // max(1, len(val_pids)))
    val_files = [p for pid in val_pids for p in val_patient_map[pid][:quota_val]][:DRY_RUN_SAMPLES]
    quota_test = max(1, DRY_RUN_SAMPLES // max(1, len(test_pids)))
    test_files = [p for pid in test_pids for p in test_patient_map[pid][:quota_test]][:DRY_RUN_SAMPLES]

print(f"[EXP 1] Running Val-to-Test Protocol: {len(val_files)} Val, {len(test_files)} Test...")
val_to_test_results = run_val_to_test_scientific_protocol(
    engine=WSSS_ENGINE,
    model_stage2=MODEL_STAGE2,
    val_files=val_files,
    test_files=test_files,
    val_mask_dir=val_mask_dir,
    test_mask_dir=test_mask_dir,
    method_key="M6_MultiScale_Pyramid_D4_Ensemble",
    model_stage1=MODEL_STAGE1,
    device=ENV_INFO["device"],
    output_dir=OUTPUT_DIR / "val_to_test",
)

print("=" * 65)
print("KET QUA QUY TRINH KHOA HOC VAL-TO-TEST PROTOCOL:")
val_miou = val_to_test_results["val_metrics"].get("mIoU_4cls", val_to_test_results["val_metrics"].get("mIoU", 0.0))
test_miou = val_to_test_results["test_metrics"].get("mIoU_4cls", val_to_test_results["test_metrics"].get("mIoU", 0.0))
gap = val_to_test_results["generalization_gap"]["mIoU_gap"] if isinstance(val_to_test_results["generalization_gap"], dict) else val_to_test_results["generalization_gap"]
print(f" - Val mIoU:            {val_miou:.2f}%")
print(f" - Test mIoU:           {test_miou:.2f}%")
print(f" - Generalization Gap:  {gap:+.2f}%")
print(f" - Test FwIoU:          {val_to_test_results['test_metrics'].get('FwIoU', 0.0):.2f}%")
print(f" - Test Boundary F1:    {val_to_test_results['test_metrics'].get('Boundary_F1', 0.0):.2f}%")
print("=" * 65)

---
## Buoc 5: Thi Nghiem 2 - Multi-Magnification Degradation & Coarse-to-Fine Pareto Frontier
Mo phong suy giam do phan giai da tang ($5\times, 10\times, 20\times, 40\times$), router do phuc tap thong minh:
$$S_{\text{route}} = 0.50 \bar{H} + 0.30 R_{\text{bound}} + 0.20 P_{\text{tumor}}$$
Doi sanh duong cong toi uu Pareto giua Adaptive Routing voi Random Routing tren cac muc ngan sach tinh toan $\beta \in [0.0, 1.0]$.

In [ ]:
import json
from src.colab_pkg.evaluator import run_multimag_and_pareto_benchmark
from src.colab_pkg.visualizer import plot_coarse_to_fine_pareto_curve

pareto_test_files = test_files[: (DRY_RUN_SAMPLES if SANITY_DRY_RUN else 200)]
print(f"[EXP 2] Running Multi-Mag & Pareto Benchmark on {len(pareto_test_files)} patches...")
pareto_data = run_multimag_and_pareto_benchmark(
    engine=WSSS_ENGINE,
    model_stage2=MODEL_STAGE2,
    test_files=pareto_test_files,
    test_mask_dir=test_mask_dir,
    model_stage1=MODEL_STAGE1,
    device=ENV_INFO["device"],
    budget_steps=5,
)

fig_pareto, pareto_img_path = plot_coarse_to_fine_pareto_curve(
    pareto_data=pareto_data,
    output_dir=OUTPUT_DIR / "pareto",
)
print(f"[OK] Pareto Frontier AUC Gain: +{pareto_data['auc_gain']:.2f}% mIoU*Budget")
print(f"[OK] Pareto chart exported: {pareto_img_path}")

# Export pareto_data to JSON: OUTPUT_DIR / "pareto" / "pareto_results.json"
def _serialize_for_json(obj, visited=None):
    if visited is None:
        visited = set()
    oid = id(obj)
    if oid in visited:
        return None
    if hasattr(obj, "item"):
        return obj.item()
    if hasattr(obj, "tolist"):
        return obj.tolist()
    if isinstance(obj, Path):
        return str(obj)
    if isinstance(obj, (int, float, str, bool)) or obj is None:
        return obj
    if isinstance(obj, dict):
        visited.add(oid)
        res = {str(k): _serialize_for_json(v, visited) for k, v in obj.items() if k != "pareto_curve"}
        visited.remove(oid)
        return res
    if isinstance(obj, (list, tuple)):
        visited.add(oid)
        res = [_serialize_for_json(v, visited) for v in obj]
        visited.remove(oid)
        return res
    return str(obj)

pareto_json_path = OUTPUT_DIR / "pareto" / "pareto_results.json"
pareto_json_path.parent.mkdir(parents=True, exist_ok=True)
with open(pareto_json_path, "w", encoding="utf-8") as f:
    json.dump(_serialize_for_json(pareto_data), f, indent=2)
print(f"[OK] Exported Pareto results to JSON: {pareto_json_path}")

---
## Buoc 6: Thi Nghiem 3 - Patient-Level Clinical Biomarker Correlation (VTP & TSR)
Luu tru va gom cum cac vung phan doan theo Patient ID:
- **VTP (Volume of Tumor Proportion)**: Ty le phan tram the tich khoi u xam lan.
- **TSR (Tumor-to-Stroma Ratio)**: Ty le U tren Mo dem - chi so tien luong doc lap chuan CAP/ASCO.
- Tinh toan he so tuong quan Pearson $r$, Spearman $\rho$, hoi quy Bland-Altman va do phu hop phan tang nguy co CAP.

In [ ]:
from collections import defaultdict
from PIL import Image
import json
import numpy as np
from src.colab_pkg.downloader import extract_patient_id
from src.colab_pkg.evaluator import run_biomarker_correlation_study, sanitize_for_json
from src.colab_pkg.visualizer import plot_biomarker_bland_altman_and_regression

patient_preds = defaultdict(list)
patient_gts = defaultdict(list)

unique_test_pids = sorted(list(set(extract_patient_id(p.name) for p in test_files)))
target_bio_count = DRY_RUN_SAMPLES if SANITY_DRY_RUN else 150
quota_bio = max(1, target_bio_count // max(1, len(unique_test_pids)))
bio_files = []
for pid in unique_test_pids:
    pid_patches = [f for f in test_files if extract_patient_id(f.name) == pid]
    bio_files.extend(pid_patches[:quota_bio])
bio_files = bio_files[:target_bio_count]

print(f"[EXP 3] Aggregating Patient Biomarkers across {len(bio_files)} patches ({len(unique_test_pids)} distinct patients)...")
for p in bio_files:
    pid = extract_patient_id(p.name)
    mask_path = test_mask_dir / p.name
    if not mask_path.is_file(): continue
    gt_arr = np.array(Image.open(mask_path))
    rgb_np = np.array(Image.open(p).convert("RGB"))
    pred_arr = WSSS_ENGINE.predict_m6(MODEL_STAGE2, None, raw_rgb=rgb_np, model_stage1=MODEL_STAGE1)
    patient_preds[pid].append(pred_arr)
    patient_gts[pid].append(gt_arr)

bio_results = run_biomarker_correlation_study(
    patient_predictions=patient_preds,
    patient_ground_truths=patient_gts,
    output_csv_path=OUTPUT_DIR / "biomarkers" / "patient_biomarker_correlation.csv",
)
fig_bio, bio_img_path = plot_biomarker_bland_altman_and_regression(
    pred_biomarkers=bio_results["pred_biomarkers"],
    gt_biomarkers=bio_results["gt_biomarkers"],
    output_dir=OUTPUT_DIR / "biomarkers",
)
corr = bio_results["correlation_metrics"]
bio_json_path = OUTPUT_DIR / "biomarkers" / "biomarker_correlation.json"
bio_json_path.parent.mkdir(parents=True, exist_ok=True)
with open(bio_json_path, "w", encoding="utf-8") as f:
    json.dump(sanitize_for_json(corr), f, indent=2)
print(f"[OK] Exported Biomarker Correlation results: {bio_json_path}")
print("=" * 65)
print(f"KET QUA TUONG QUAN CHI SO SINH HOC LAM SANG ({corr['patient_count']} Patients):")
print(f" - VTP Pearson r:         {corr['vtp_pearson_r']:.4f} (p = {corr['vtp_pearson_p']:.4e})")
print(f" - TSR Pearson r:         {corr['tsr_pearson_r']:.4f} (p = {corr['tsr_pearson_p']:.4e})")
print(f" - VTP Spearman rho:      {corr['vtp_spearman_rho']:.4f}")
print(f" - CAP Risk Concordance:  {corr['cap_risk_concordance_pct']:.2f}%")
print("=" * 65)

---
## Buoc 7: Thi Nghiem 4 - Cross-Dataset LUAD-HistoSeg Generalization (Ung Thu Phoi)
Kiem dinh kha nang tong quat hoa khac co quan (Cross-Organ Zero-Shot):
- Tai dataset LUAD-HistoSeg (Lung Adenocarcinoma, 4 classes: Tumor, Stroma, Normal, Necrosis).
- Danh gia mIoU, Dice va FwIoU de do luong do suy giam hieu nang khi chuyen doi tu Ung thu vu sang Ung thu phoi.

In [ ]:
from src.colab_pkg.downloader import ensure_luad_checkpoints, ensure_luad_dataset
from src.colab_pkg.evaluator import run_cross_dataset_luad_benchmark
from src.colab_pkg.model_loader import load_luad_models

try:
    luad_root, luad_img_dir, luad_mask_dir = ensure_luad_dataset(data_root=WORK_ROOT / "LUAD-HistoSeg", drive_cache_dir=None)
    luad_ckpts = ensure_luad_checkpoints(checkpoint_dir=WORK_ROOT / "checkpoints", drive_cache_dir=None)
    model_luad_s2, model_luad_s1, _ = load_luad_models(
        stage2_ckpt=luad_ckpts.get("stage2"),
        stage1_ckpt=luad_ckpts.get("stage1"),
        device=ENV_INFO["device"],
    )
    luad_test_files = sorted(luad_img_dir.glob("*.png"))
    if SANITY_DRY_RUN:
        luad_test_files = luad_test_files[:20]

    print(f"[EXP 4] Running LUAD Cross-Dataset Benchmark on {len(luad_test_files)} patches...")
    luad_results = run_cross_dataset_luad_benchmark(
        engine=WSSS_ENGINE,
        model_stage2=model_luad_s2,
        test_files=luad_test_files,
        test_mask_dir=luad_mask_dir,
        model_stage1=model_luad_s1,
        device=ENV_INFO["device"],
        output_dir=OUTPUT_DIR / "luad",
    )
    l_miou = luad_results.get("mIoU_4cls", luad_results.get("mIoU", 0.0))
    l_dice = luad_results.get("mDice_4cls", luad_results.get("mDice", 0.0))
    print(f"[OK] LUAD Test mIoU: {l_miou:.2f}% | Dice: {l_dice:.2f}%")
except Exception as e:
    print(f"[NOTE] Bo qua LUAD benchmark hoac loi tai nguyen: {e}")

---
## Buoc 8: Thi Nghiem 5 - Whole-ROI 100% Tissue Coverage Evaluation
Kiem dinh tren cac tieu ban Whole-ROI co Ground-Truth chu thich day du tu bac si giai phau benh (BCSS Annotated Cohort):
- Khoi tao canvas voi do bao phu 100% dien tich mo tu quet Low-Mag thong qua Coarse Base Map.
- Tinh toan mIoU, Dice, Pixel Accuracy, FwIoU va Boundary F1 tren toan bo vung ROI.

In [ ]:
import json
import numpy as np
from PIL import Image
from src.colab_pkg.downloader import download_bcss_whole_roi_cohort
from src.colab_pkg.evaluator import sanitize_for_json
from src.colab_pkg.visualizer import plot_whole_roi_ground_truth_comparison
from src.pipeline.wsi_lowmag_highmag_pipeline import WSILowMagHighMagPipeline

pipeline_wsi = WSILowMagHighMagPipeline(
    patch_size=224,
    lowmag_target_size=(512, 512),
    tissue_threshold=0.15,
    engine=WSSS_ENGINE,
    device=ENV_INFO["device"],
)

try:
    roi_cohort = download_bcss_whole_roi_cohort(dest_dir=WORK_ROOT / "bcss_whole_roi")
    if roi_cohort and len(roi_cohort) > 0:
        sample_roi = roi_cohort[0]
        roi_img = np.array(Image.open(sample_roi["img"]).convert("RGB"))
        roi_gt = np.array(Image.open(sample_roi["mask"]))
        stitched_pred, rep = pipeline_wsi.run_end_to_end_wsi(
            slide_input=roi_img,
            model_stage2=MODEL_STAGE2,
            model_stage1=MODEL_STAGE1,
            slide_id=sample_roi.get("patient_id", "WHOLE_ROI_TEST"),
            max_patches_to_infer=50 if SANITY_DRY_RUN else None,
        )
        roi_metrics = pipeline_wsi.evaluate_wsi_with_ground_truth_roi(stitched_pred, roi_gt)
        fig_roi, roi_fig_path = plot_whole_roi_ground_truth_comparison(
            rgb_image=roi_img, pred_mask=stitched_pred, gt_mask=roi_gt, output_dir=OUTPUT_DIR / "whole_roi",
        )
        whole_roi_json = OUTPUT_DIR / "whole_roi" / "whole_roi_metrics.json"
        whole_roi_json.parent.mkdir(parents=True, exist_ok=True)
        with open(whole_roi_json, "w", encoding="utf-8") as f:
            json.dump(sanitize_for_json(roi_metrics), f, indent=2)
        print(f"[OK] Exported Whole-ROI metrics: {whole_roi_json}")
        print(f"[OK] Whole-ROI mIoU: {roi_metrics['mIoU_4cls']:.2f}% | FwIoU: {roi_metrics['FwIoU']:.2f}%")
    else:
        print("[NOTE] Khong tim thay mau Whole-ROI trong thu muc.")
except Exception as e:
    print(f"[NOTE] Bo qua Whole-ROI test: {e}")

---
## Buoc 9: Dong Goi Ket Qua & Tu Dong Tai Truc Tiep Ve May Tinh (Direct Browser Download)
Toan bo ket qua nghien cuu khoa hoc (cac file JSON chi so, bang tong hop CSV, bieu do xuat ban 300-DPI chat luong cao PNG) se duoc tu dong nen thanh goi ZIP va kich hoat trinh duyet tai truc tiep ve may tinh cua ban ma hoan toan khong can Google Drive.

In [ ]:
import os, sys, json, zipfile, shutil
from pathlib import Path
from src.colab_pkg.evaluator import sanitize_for_json

# Export consolidated master benchmark summary JSON before zipping
master_summary = {
    "project": "SE374.R11 Medical WSSS Scientific Research Runner",
    "device": str(ENV_INFO.get("device", "cpu")),
    "sanity_dry_run": SANITY_DRY_RUN,
    "val_to_test": val_to_test_results if "val_to_test_results" in locals() else {},
    "pareto": pareto_data if "pareto_data" in locals() else {},
    "biomarkers": bio_results if "bio_results" in locals() else {},
    "luad_generalization": luad_results if "luad_results" in locals() else {},
    "whole_roi": roi_metrics if "roi_metrics" in locals() else {},
}
summary_json_path = OUTPUT_DIR / "scientific_benchmark_summary.json"
summary_json_path.parent.mkdir(parents=True, exist_ok=True)
with open(summary_json_path, "w", encoding="utf-8") as f:
    json.dump(sanitize_for_json(master_summary), f, indent=2, default=str)
print(f"[OK] Exported master benchmark summary: {summary_json_path}")

zip_name = "SE374_Scientific_Research_Results.zip"
zip_out = WORK_ROOT / zip_name
if zip_out.is_file():
    zip_out.unlink()

print(f"--> Dang dong goi ket qua nghien cuu tu {OUTPUT_DIR} vao file ZIP...")
with zipfile.ZipFile(zip_out, "w", zipfile.ZIP_DEFLATED) as zf:
    for root, _, files in os.walk(OUTPUT_DIR):
        for f in files:
            p = Path(root) / f
            zf.write(p, arcname=p.relative_to(OUTPUT_DIR))

zip_size_mb = zip_out.stat().st_size / (1024 * 1024)
print(f"[OK] Da tao thanh cong goi ket qua: {zip_out.resolve()} ({zip_size_mb:.2f} MB)")

if "google.colab" in sys.modules:
    try:
        from google.colab import files
        print("--> Dang kich hoat hop thoai tai ve tren trinh duyet cua ban...")
        files.download(str(zip_out))
        print("[THANH CONG] Trinh duyet dang tai tap tin ket qua ve may tinh cua ban!")
    except Exception as err:
        print(f"[HUONG DAN] Neu trinh duyet chan popup tu dong, ban co the mo thanh quan ly file ben trai Colab va tai file '{zip_name}'.")
else:
    print(f"[OK] Ket qua duoc luu tai thu muc cuc bo: {zip_out.resolve()}")